# The Cost of Utilities, 2020–2025

**Task:** analyse the monthly cost (USD) of basic utilities (electricity, heating, cooling, water and waste removal for an 85 m² apartment) in capital cities across Europe, the Middle East, Asia, North America, South America, Oceania and Africa.

**Data:** Numbeo's *Basic utilities* series, the figure quoted in Deutsche Bank Research's annual *Mapping the World's Prices* reports (and charted by Visual Capitalist). Inflation from the IMF World Economic Outlook; CPI index and exchange rates from the World Bank.

**Questions**
1. How have utility bills moved since 2020, by region?
2. Which capitals pay the most and the least?
3. Is the increase in line with inflation?
4. What is driving the change in the dollar bill: local prices or exchange rates?
5. When did the increases hit (the 2022 energy shock)?
6. *Extra:* do high-inflation countries hold utilities down (a sign of regulated tariffs/subsidies)?

The interactive version of this analysis is in `dashboard/index.html`.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import analysis

DATA_DIR = os.environ.get("UTILITIES_DATA_DIR", analysis.DATA_DIR)
sns.set_theme(style="whitegrid", context="notebook")
REGION_COLORS = dict(zip(analysis.REGIONS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6"]))
pd.options.display.float_format = "{:,.1f}".format

## 1. Get the data
Run once (needs network access to numbeo.com, imf.org and api.worldbank.org). It writes `data/utilities_monthly_usd.csv` and `data/macro.csv` and lists any gaps to fill by hand.

In [ ]:
# !python fetch_data.py

## 2. Load and check coverage

In [ ]:
df = analysis.load(DATA_DIR)
if df.empty:
    raise SystemExit("No utilities data yet: run fetch_data.py (or fill data/utilities_monthly_usd.csv) first.")

coverage = df.pivot_table(index=["region", "city"], columns="year", values="utilities_usd")
print(f"{df.city.nunique()} cities, {df.year.min()}–{df.year.max()}, {coverage.isna().sum().sum()} missing city-years")
missing_macro = df[df[["cpi_level", "fx_lcu_per_usd"]].isna().any(axis=1)][["city", "year"]]
if len(missing_macro):
    print("Missing inflation/FX for:", missing_macro.to_dict("records"))
coverage

In [ ]:
summary = analysis.city_summary(df)
summary[["city", "region", "start_usd", "end_usd", "usd_change_pct", "lcu_change_pct",
         "local_inflation_pct", "real_change_pct", "fx_change_pct", "peak_year"]]

## 3. Q1: How have bills moved since 2020?
Median city per region, indexed to 2020 = 100 in USD, against US inflation.

In [ ]:
reg = analysis.region_series(df)
us = df.drop_duplicates("year").set_index("year").us_cpi_level
fig, ax = plt.subplots(figsize=(11, 5))
for region, g in reg.groupby("region"):
    ax.plot(g.year, g.usd_index, lw=2, color=REGION_COLORS[region], label=region)
    ax.annotate(f"{region} {g.usd_index.iloc[-1]:.0f}", (g.year.iloc[-1], g.usd_index.iloc[-1]),
                xytext=(6, 0), textcoords="offset points", va="center", fontsize=9)
ax.plot(us.index, 100 * us / us.iloc[0], ls=":", lw=2, color="grey", label="US inflation")
ax.axhline(100, color="#c3c2b7", lw=1)
ax.set(title="Utility bills by region (USD, 2020 = 100)", xlabel="", ylabel="Index")
ax.legend(ncol=4, frameon=False, loc="upper left")
sns.despine(); plt.show()

## 4. Q2: Who pays the most?

In [ ]:
latest = df[df.year == df.year.max()].sort_values("utilities_usd", ascending=False)
fig, ax = plt.subplots(figsize=(9, 8))
y = np.arange(len(latest))[::-1]
ax.barh(y, latest.utilities_usd, color=latest.region.map(REGION_COLORS), height=0.7)
for yi, v in zip(y, latest.utilities_usd):
    ax.text(v + 2, yi, f"${v:,.0f}", va="center", fontsize=9)
ax.set_yticks(y, latest.city)
for region, color in REGION_COLORS.items():
    ax.barh([], [], color=color, label=region)
ax.set(title=f"Monthly basic utilities, {df.year.max()} (USD, 85 m² apartment)", xlabel="USD per month", ylabel="")
ax.legend(frameon=False); sns.despine(); plt.show()

latest.groupby("region").utilities_usd.agg(["mean", "min", "max"]).sort_values("mean", ascending=False)

## 5. Q3: Is it in line with inflation?
Compare the bill's change **in local currency** with local CPI inflation. That removes exchange-rate noise and asks the question residents care about: did utilities get more expensive relative to everything else?

In [ ]:
s = summary.sort_values("real_change_pct")
fig, ax = plt.subplots(figsize=(9, 9))
y = np.arange(len(s))
ax.hlines(y, s.local_inflation_pct, s.lcu_change_pct, color="#c3c2b7", lw=2)
ax.scatter(s.local_inflation_pct, y, color="#2a78d6", s=40, label="Local CPI inflation", zorder=3)
ax.scatter(s.lcu_change_pct, y, color="#eb6834", s=40, label="Utilities (local currency)", zorder=3)
ax.set_yticks(y, s.city)
if s[["lcu_change_pct", "local_inflation_pct"]].abs().max().max() > 400:
    ax.set_xscale("symlog", linthresh=50)
ax.axvline(0, color="#c3c2b7", lw=1)
ax.set(title="Cumulative change 2020–latest: utilities vs inflation", xlabel="% change")
ax.legend(frameon=False, loc="lower right"); sns.despine(); plt.show()

print(f"{summary.beat_inflation.sum()} of {len(summary)} capitals saw utilities rise faster than local inflation.")
summary.groupby("region")[["lcu_change_pct", "local_inflation_pct", "real_change_pct"]].median()

## 6. Q4: What drives the change in the USD bill?
In log points the USD change splits exactly:

$$\ln\frac{USD_t}{USD_0} = \underbrace{\ln\frac{CPI_t}{CPI_0}}_{\text{local inflation}} + \underbrace{\ln\frac{LCU_t / LCU_0}{CPI_t / CPI_0}}_{\text{utilities vs inflation}} - \underbrace{\ln\frac{FX_t}{FX_0}}_{\text{currency vs USD}}$$

A currency that weakens against the dollar pulls the USD bill down even when local bills soar, so a falling dollar figure does not mean residents pay less.

In [ ]:
parts = {"lp_inflation": ("Local inflation", "#2a78d6"), "lp_real": ("Utilities vs inflation", "#eb6834"),
         "lp_fx": ("Currency vs USD", "#1baf7a")}
s = summary.sort_values("lp_total")
fig, ax = plt.subplots(figsize=(10, 9))
pos = np.zeros(len(s)); neg = np.zeros(len(s)); y = np.arange(len(s))
for col, (name, color) in parts.items():
    v = s[col].values
    left = np.where(v > 0, pos, neg)
    ax.barh(y, v, left=left, color=color, label=name, height=0.6)
    pos += np.clip(v, 0, None); neg += np.clip(v, None, 0)
ax.scatter(s.lp_total, y, color="black", zorder=3, s=25, label="Total USD change")
ax.set_yticks(y, s.city); ax.axvline(0, color="black", lw=0.8)
ax.set(title="Decomposition of the USD change (log points ≈ %)", xlabel="log points")
ax.legend(frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.07)); sns.despine(); plt.show()

drivers = s.assign(main_driver=s[list(parts)].abs().idxmax(axis=1).map(lambda c: parts[c][0]))
drivers[["city", "usd_change_pct", "lp_inflation", "lp_real", "lp_fx", "main_driver"]].sort_values("usd_change_pct", ascending=False)

## 7. Q5: When did the increases hit?
Year-on-year change in the USD bill. Look for a 2022 spike in Europe (gas crisis) and for years where subsidies were withdrawn.

In [ ]:
yoy = df.pivot_table(index=["region", "city"], columns="year", values="usd_yoy_pct").dropna(axis=1, how="all")
fig, ax = plt.subplots(figsize=(9, 10))
sns.heatmap(yoy, cmap="RdBu_r", center=0, vmin=-40, vmax=40, annot=True, fmt=".0f", cbar_kws={"label": "% YoY"}, ax=ax)
ax.set(title="Year-on-year change in the USD bill (%)", xlabel="", ylabel=""); plt.show()

print("Median YoY change by year:"); print(yoy.median().round(1).to_string())
print("Most common peak year:", summary.peak_year.mode().iloc[0])

## 8. Extra: do high-inflation countries hold utilities down?
If governments cap tariffs when inflation is high, utilities fall behind CPI where inflation is highest: a negative slope below.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=summary, x="local_inflation_pct", y="real_change_pct", hue="region", palette=REGION_COLORS, s=70, ax=ax)
for _, r in summary.iterrows():
    ax.annotate(r.city, (r.local_inflation_pct, r.real_change_pct), xytext=(4, 3), textcoords="offset points", fontsize=8)
if summary.local_inflation_pct.max() > 400:
    ax.set_xscale("symlog", linthresh=50)
ax.axhline(0, color="#c3c2b7", lw=1)
ax.set(xlabel="Cumulative local inflation (%)", ylabel="Utilities vs inflation (%)", title="Inflation vs real change in utility bills")
ax.legend(frameon=False, fontsize=8); sns.despine(); plt.show()

rho = summary[["local_inflation_pct", "real_change_pct"]].corr(method="spearman").iloc[0, 1]
print(f"Spearman correlation: {rho:.2f}")

## 9. Findings
Computed headline facts (they drive the dashboard's answers too):

In [ ]:
findings = analysis.findings(df, summary)
findings

**Narrative** *(write this once the real data is loaded; the numbers above are the evidence)*

1. **Overall trend:** …
2. **Versus inflation:** …
3. **Drivers:** energy shock / subsidies / currency moves …
4. **Regional differences:** …
5. **Caveats:** Numbeo figures are crowd-sourced; small samples can swing a single year. USD figures reflect exchange rates as well as prices.

## 10. Build the interactive dashboard

In [ ]:
path = analysis.export_dashboard(df)
print(f"Wrote {path}. Open dashboard/index.html in a browser.")